# Linear Attention — NumPy 실습
논문: https://arxiv.org/pdf/2006.16236

일반 커널 Attention과 결합법칙을 활용한 Linear Attention을 비교합니다. **Non-causal** 예제이며 Softmax Attention과 같다는 뜻은 아닙니다.

In [1]:
import numpy as np
rng = np.random.default_rng(42)
N, D, M = 4, 3, 2
Q, K, V = rng.normal(size=(N,D)), rng.normal(size=(N,D)), rng.normal(size=(N,M))
def phi(x):
    return np.where(x > 0, x+1, np.exp(x)) # ELU(x)+1; elementwise
print(Q.shape, K.shape, V.shape)

(4, 3) (4, 3) (4, 2)


## 1. 직접 계산
A = phi(Q) @ phi(K).T 는 NxN 유사도 행렬입니다. 각 행을 합치는 `sum(axis=1)`이 rowwise sum입니다.

In [3]:
qf, kf = phi(Q), phi(K)
A = qf @ kf.T
denom = A.sum(axis=1)  # rowwise sum
out_direct = (A @ V) / denom[:, None]
print('A:\n', A.round(3))
print('row sums:', denom.round(3))
print('output:\n', out_direct.round(3))

A:
 [[4.712 1.707 4.242 2.946]
 [2.77  1.121 4.006 1.531]
 [4.196 1.852 3.629 3.033]
 [7.061 3.434 4.066 5.916]]
row sums: [13.607  9.428 12.711 20.477]
output:
 [[ 0.511 -0.03 ]
 [ 0.461  0.057]
 [ 0.565 -0.037]
 [ 0.642 -0.092]]


## 2. Linear Attention
결합법칙으로 `(phi(Q) @ phi(K).T) @ V` 대신 `phi(Q) @ (phi(K).T @ V)`를 계산합니다. NxN 행렬을 만들지 않습니다.

In [5]:
S = kf.T @ V   # (D,M)
z = kf.sum(axis=0) # (D,)
out_linear = (qf @ S) / (qf @ z)[:, None]
print('S shape:',S.shape, 'z shape:',z.shape)
print('linear result:\n',out_linear.round(3))
print('같은 결과?', np.allclose(out_direct, out_linear))

S shape: (3, 2) z shape: (3,)
linear result:
 [[ 0.511 -0.03 ]
 [ 0.461  0.057]
 [ 0.565 -0.037]
 [ 0.642 -0.092]]
같은 결과? True


## 3. Softmax Attention과 비교
일반 Softmax Attention은 다른 유사도 함수이므로 결과가 같지 않습니다.

In [6]:
scores = Q @ K.T / np.sqrt(D)
exp_scores = np.exp(scores - scores.max(axis=1,keepdims=True))
out_softmax = (exp_scores / exp_scores.sum(axis=1,keepdims=True)) @ V
print('softmax:\n',out_softmax.round(3))
print('linear kernel:\n',out_linear.round(3))

softmax:
 [[ 0.495  0.101]
 [ 0.496  0.312]
 [ 0.621  0.054]
 [ 0.818 -0.165]]
linear kernel:
 [[ 0.511 -0.03 ]
 [ 0.461  0.057]
 [ 0.565 -0.037]
 [ 0.642 -0.092]]


## 4. 계산량 비교
직접 계산은 대략 `O(N²(D+M))`, 재배열된 Linear Attention은 `O(NDM+ND)`입니다. D,M이 고정되면 N에 선형입니다.

In [15]:
from time import perf_counter

for _ in range(1000):
    continue

for n in (999, 64,256,1024):
    q,k,v = rng.normal(size=(n,32)),rng.normal(size=(n,32)),rng.normal(size=(n,32))
    qf,kf = phi(q),phi(k)
    t=perf_counter(); a=qf@kf.T; direct=(a@v)/a.sum(axis=1,keepdims=True); dt=perf_counter()-t
    t=perf_counter(); linear=(qf@(kf.T@v))/(qf@kf.sum(axis=0))[:,None]; lt=perf_counter()-t
    assert np.allclose(direct,linear)
    if n != 999:
      print(f'N={n} direct={dt*1e3:.2f}ms linear={lt*1e3:.2f}ms')

N=64 direct=0.92ms linear=0.12ms
N=256 direct=0.57ms linear=0.12ms
N=1024 direct=5.17ms linear=0.45ms
